# Badge 3: Data Application Builders Workshop: Lesson 6: Pending Orders App Improvements

The order sequence, ownership audit, the final ORDERS definition and the merge.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE3_DABW_ADMIN` (ACCOUNTADMIN), `BADGE3_DABW_ROLE` (SYSADMIN) and `BADGE3_DABW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE3_DABW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE3_DABW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE3_DABW_WH"      # stands in for the course's COMPUTE_WH
import ast, io
from snowflake.snowpark.context import get_active_session

CUSTOM = 'SMOOTHIES.PUBLIC."Custom Smoothie Order Form"'
PENDING = 'SMOOTHIES.PUBLIC."Pending Smoothie Orders"'


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result


def write_app(app, app_code):
    """SQL equivalent of editing the app in Snowsight's Streamlit editor: write streamlit_app.py into the app's live version."""
    ast.parse(app_code)  # the code must at least be valid Python
    path = f"snow://streamlit/{app}/versions/live/streamlit_app.py"
    get_active_session().file.put_stream(io.BytesIO(app_code.encode()), path, auto_compress=False, overwrite=True)
    print(f"wrote {len(app_code.splitlines())} lines to {path}")

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: ❄️ Ensuring Uniqueness

> **Course:** Create a Sequence to Use as a Row ID (screenshots 2026011612105, 111-1): ORDER_SEQ in SMOOTHIES.PUBLIC, created as SYSADMIN.

In [ ]:
%%sql -r create_order_seq
-- From screenshot 2026011612111-1
use role {{SYSADMIN_ROLE}};
create sequence smoothies.public.order_seq
    start = 1
    increment = 2
    ORDER
    comment = 'Provide a unique id for each smoothie order';

> **Course:** Truncate the Orders Table, then add the unique ID column (screenshot 2026011612111).

In [ ]:
%%sql -r truncate_orders
truncate table SMOOTHIES.PUBLIC.ORDERS;

In [ ]:
%%sql -r add_order_uid
alter table SMOOTHIES.PUBLIC.ORDERS 
add column order_uid integer --adds the column
default smoothies.public.order_seq.nextval  --sets the value of the column to sequence
constraint order_uid unique enforced; --makes sure there is always a unique value in the column

## Page: 🔐 Checking All Objects for SYSADMIN Ownership

> **Course:** Audit Ownership of All Your Objects: make sure they are all owned by the SYSADMIN role (video).

In [ ]:
%%sql -r ownership_audit
-- Derived solution (the course shows a video): owners of the objects created so far
select 'DATABASE' as kind, database_name as name, database_owner as owner from smoothies.information_schema.databases where database_name = 'SMOOTHIES'
union all select 'SCHEMA', schema_name, schema_owner from smoothies.information_schema.schemata where schema_name = 'PUBLIC'
union all select 'TABLE', table_name, table_owner from smoothies.information_schema.tables where table_schema = 'PUBLIC'
union all select 'STAGE', stage_name, stage_owner from smoothies.information_schema.stages
union all select 'FILE FORMAT', file_format_name, file_format_owner from smoothies.information_schema.file_formats where file_format_schema = 'PUBLIC'
union all select 'SEQUENCE', sequence_name, sequence_owner from smoothies.information_schema.sequences;

In [ ]:
%%sql -r streamlit_owners
show streamlit apps in schema SMOOTHIES.PUBLIC;

> **Course:** Test Your Apps: submit a few orders in the ordering app and make sure they arrive in the Pending Orders app.

In [ ]:
%%sql -r submit_test_orders
insert into smoothies.public.orders(ingredients, name_on_order) values ('Dragon Fruit Honeydew Guava Tangerine ','Betty Jean');
select * from smoothies.public.orders;

## Page: 🧾 Perfecting the ORDERS Table Definition

> **Course:** Drop and recreate the table with the course's definition (this also gives a fresh start on orders).

In [ ]:
%%sql -r recreate_orders
create or replace table smoothies.public.orders (
       order_uid integer default smoothies.public.order_seq.nextval,
       order_filled boolean default false,
       name_on_order varchar(100),
       ingredients varchar(200),
       constraint order_uid unique (order_uid),
       order_ts timestamp_ltz default current_timestamp()
);

In [ ]:
%%sql -r check_orders_owner
show tables like 'ORDERS' in schema SMOOTHIES.PUBLIC;

> **Course:** Test Your SiS Apps Again (screenshots 2026011612213, 213-1): submit an order, then mark it filled.

In [ ]:
%%sql -r submit_again
insert into smoothies.public.orders(ingredients, name_on_order) values ('Dragon Fruit Honeydew Guava Tangerine ','Betty Jean');

> **Course:** Add a Merge Statement (course code, inside `if submitted:`; screenshot 2026011612224 shows the lines). `when_matched` is imported with `col`.

In [ ]:
write_app(PENDING, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, when_matched

# Write directly to the app
st.title(":cup_with_straw: Pending Smoothie Orders :cup_with_straw:")
st.write("Orders that need to be filled.")

session = get_active_session()
my_dataframe = session.table("smoothies.public.orders") \
    .filter(col("ORDER_FILLED") == 0) \
    .collect()
editable_df = st.data_editor(my_dataframe)

submitted = st.button('Submit')

if submitted:
    st.success("Someone clicked the button.", icon="👍")

    og_dataset = session.table("smoothies.public.orders")
    edited_dataset = session.create_dataframe(editable_df)
    og_dataset.merge(edited_dataset
                     , (og_dataset['ORDER_UID'] == edited_dataset['ORDER_UID'])
                     , [when_matched().update({'ORDER_FILLED': edited_dataset['ORDER_FILLED']})]
                    )
''')

> ⚠️ **Note:** The course's merge statement is run by clicking Submit in the app; the SQL equivalent of ticking ORDER_FILLED for an order and clicking Submit follows.

In [ ]:
%%sql -r tick_filled
-- SQL equivalent of ticking ORDER_FILLED and clicking Submit (the merge matches rows by ORDER_UID)
merge into smoothies.public.orders o
using (select order_uid, true as order_filled from smoothies.public.orders where name_on_order = 'Betty Jean') e
on o.order_uid = e.order_uid
when matched then update set order_filled = e.order_filled;

## Page: 🧾 Improving the Pending Orders App

> **Course:** Making the Success Message More Meaningful (try/except around the merge) and Making the GUI Look Better (no table or Submit button when there are no pending orders); screenshots 2026011612224, 230, 231.

In [ ]:
write_app(PENDING, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, when_matched

# Write directly to the app
st.title(":cup_with_straw: Pending Smoothie Orders :cup_with_straw:")
st.write("""Orders that need to filled.""")

session = get_active_session()
my_dataframe = session.table("smoothies.public.orders") \
    .filter(col("ORDER_FILLED") == 0) \
    .collect()

if my_dataframe:
    editable_df = st.data_editor(my_dataframe)
    submitted = st.button('Submit')
    if submitted:

        og_dataset = session.table("smoothies.public.orders")
        edited_dataset = session.create_dataframe(editable_df)

        try:
            og_dataset.merge(edited_dataset
                             , (og_dataset['ORDER_UID'] == edited_dataset['ORDER_UID'])
                             , [when_matched().update({'ORDER_FILLED': edited_dataset['ORDER_FILLED']})]
                            )
            st.success("Order(s) Updated!", icon="👍")
        except:
            st.write('Something went wrong.')
else:
    st.success('There are no pending orders right now', icon="👍")
''')

In [ ]:
%%sql -r run_pending_4
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Pending Smoothie Orders"();

## Page: 🍒 To-Do #3: A Way to Limit Fruits Added

> **Course:** Limit entries with `max_selections=5` on the multiselect (screenshots 2026011612245, 250).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

name_on_order = st.text_input('Name on Smoothie:')
st.write('The name on your Smoothie will be:', name_on_order)

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    , max_selections=5
    )

if ingredients_list:
    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen + ' '

    #st.write(ingredients_string)


    my_insert_stmt = """ insert into smoothies.public.orders(ingredients, name_on_order)
            values ('""" + ingredients_string + """','"""+name_on_order+ """')"""

    # st.write(my_insert_stmt)
    # st.stop()

    time_to_insert = st.button('Submit Order')

    if time_to_insert:
        session.sql(my_insert_stmt).collect()

        st.success(f'Your Smoothie is ordered, {name_on_order}!', icon="✅")
''')

In [ ]:
%%sql -r run_app_6
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

## Page: 🟢 Smoothie Fridays Are A GO

Story only.

## Page: 🤖 DORA DABW005

### 🤖 DORA check DABW005 goes here
The course has the learner run **DABW005** (two SiS apps (EXECUTE STREAMLIT queries) exist in query history) at this point.
It is in `dora/dora_tests.ipynb`, section **DABW005**, as a local check plus the exact DORA code.